# Fake Email Detection: Supervised Learning Model Comparison

Is notebook mein CSV dataset par 14 common supervised text-classification algorithms train hote hain. Sab models ek hi stratified 75/25 train/test split aur TF-IDF features use karte hain, taaki accuracy fair tarike se compare ho. Chhote starter dataset ke scores sirf demo hain; real use ke liye kaafi bada labelled dataset chahiye.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from train_model import load_data, build_candidates

df, dataset_source = load_data()
print("Dataset:", dataset_source)
print("Labelled emails:", len(df))
display(df["label"].value_counts().rename(index={0: "Legitimate / ham", 1: "Suspicious / spam"}))
df.head()

In [ ]:
# One identical stratified split is shared by every model.
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["text"], df["label"], test_size=0.25, random_state=42, stratify=df["label"]
)
print(f"Train rows: {len(X_train_text)} | Test rows: {len(X_test_text)}")

In [ ]:
# Train every classifier using TF-IDF text features; keep fitted pipelines for prediction.
models = {}
results = []
for name, classifier in build_candidates().items():
    model = Pipeline([
        ("tfidf", TfidfVectorizer(stop_words="english", ngram_range=(1, 2), sublinear_tf=True)),
        ("classifier", classifier),
    ])
    model.fit(X_train_text, y_train)
    predictions = model.predict(X_test_text)
    score = accuracy_score(y_test, predictions) * 100
    models[name] = model
    results.append({"Algorithm": name, "Accuracy (%)": score})
    print(f"{name}: {score:.2f}%")

In [ ]:
# Compare accuracy from highest to lowest.
accuracy_table = pd.DataFrame(results).sort_values("Accuracy (%)", ascending=False).reset_index(drop=True)
display(accuracy_table.style.format({"Accuracy (%)": "{:.2f}%"}))
best_name = accuracy_table.loc[0, "Algorithm"]
best_model = models[best_name]
print("Best model on this test split:", best_name)

In [ ]:
# Detailed metrics for the top model (accuracy alone can hide missed phishing emails).
best_predictions = best_model.predict(X_test_text)
print(classification_report(y_test, best_predictions, target_names=["Legitimate", "Suspicious"], zero_division=0))

In [ ]:
# Try an example email with the highest-accuracy model from this split.
sample_email = ["Urgent: verify your account password using this link immediately"]
prediction = int(best_model.predict(sample_email)[0])
print("Model:", best_name)
print("Result:", "Suspicious / spam" if prediction == 1 else "Likely legitimate")